# 08 - RQ3: Đánh giá mô hình trên ảnh che theo mặt nạ phổi và kiểm tra Shortcut Learning (S4-07, S5-01, S5-02, S5-03)

**Bài tập lớn môn Học máy:** Phân loại 4 nhóm ảnh X-quang ngực (`COVID`, `Lung_Opacity`, `Normal`, `Viral Pneumonia`).  
**Câu hỏi nghiên cứu 3 (RQ3):** Việc che/cắt theo mặt nạ phổi có làm thay đổi Macro-F1 validation của mô hình học máy tối ưu không? Đặc trưng rút ra từ bên trong vùng phổi (`lung_only`) so với đặc trưng rút ra từ bên ngoài vùng phổi (`outside_only`) có hiệu năng như thế nào trong việc phân biệt bệnh lý và phát hiện hiện tượng học lối tắt (*shortcut learning*)?  
**Mô hình và đặc trưng đã khóa từ RQ1 & RQ2:** Nhóm đặc trưng `ghep` (152 chiều = Histogram 64 + GLCM 60 + LBP 28) + `StandardScaler` (chỉ fit trên train tương ứng) + `SVC(kernel='rbf', C=10.0, gamma='scale', class_weight='balanced', probability=True, random_state=42)` (`results/rq2_classifier_selection_record.md`).  
**Người phụ trách (Code Owner):** Lê Minh (duyệt Minh Thuận — S4-07; duyệt Cả nhóm — S5-01, S5-02, S5-03).

## Nội dung thực hiện theo `results/threshold_v3.md`:
1. **Kiểm tra trực quan & căn chỉnh mặt nạ phổi (S4-07):** Co giãn mặt nạ phổi từ `256×256` lên `299×299` bằng nội suy lân cận gần nhất (`cv2.INTER_NEAREST`, ngưỡng > 127). Kiểm tra trực quan trên 80 mẫu validation phân tầng (20 ảnh/lớp, `allow_test=False`) và tính toán các chỉ số hình học căn chỉnh giải phẫu.
2. **Kiểm toán 5 run thực nghiệm RQ3 (S5-01 & Cổng H1–H6):** Đối chiếu 5 kịch bản (`orig_to_orig`, `orig_to_lung`, `orig_to_out`, `lung_to_lung`, `out_to_out`), xác minh ngân sách tính toán (Cổng H5) và tính hợp lệ xác suất dự đoán.
3. **Xuất và trình bày bộ báo cáo bắt buộc R1, R2, R4, R5 (`src/bootstrap.py`):**
   - **R1:** Bảng Macro-F1 và Precision/Recall/F1 từng lớp kèm khoảng tin cậy 95% bootstrap (2.000 lần lấy mẫu theo ảnh).
   - **Mục 5 (Phép so sánh chính RQ3):** Kiểm định bootstrap ghép cặp 2.000 lần giữa `lung_to_lung` và tham chiếu `orig_to_orig` (ngưỡng cải thiện $\Delta \ge +0,005$, khoảng tin cậy không chứa 0).
   - **R3 (Kiểm tra lối tắt):** So sánh đối chứng Trong phổi vs Ngoài phổi (`lung_to_lung` vs `out_to_out`) và Toàn ảnh vs Ngoài phổi (`orig_to_orig` vs `out_to_out`).
   - **R2, R4, R5:** Ma trận con 2×2 `Normal – Lung_Opacity`, sai số chuẩn lấy mẫu xấp xỉ ($n = 3.174$ và $n = 203$) và giới hạn phương pháp luận.
4. **Phân tích độ quan trọng hoán vị khối (S5-02):** Đo lường mức độ sụt giảm Macro-F1 trên 10 khối đặc trưng (`hist_64`, `glcm_60`, `lbp_28`, 5 tiểu khối GLCM và 2 tiểu khối LBP) trên cả 3 mô hình (`original`, `lung_only`, `outside_only`).
5. **Kết luận về Shortcut Learning và khóa cấu hình cho Tuần 10 (S5-03):** Lập luận y sinh học và quyết định đóng băng cấu hình chính thức mở khóa tập `test` ở Tuần 10 (`S5-04`).


In [ ]:
from __future__ import annotations

import json
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import yaml

# Định vị thư mục gốc của repository
REPO_ROOT = Path.cwd()
if not (REPO_ROOT / "results" / "threshold_v3.md").exists() and (REPO_ROOT.parent / "results" / "threshold_v3.md").exists():
    REPO_ROOT = REPO_ROOT.parent
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from src.bootstrap import (
    align_predictions,
    build_rq3_report,
    load_rq3_predictions,
    macro_f1_from_labels,
    paired_bootstrap_diff,
)
from src.metrics import DEFAULT_CLASS_NAMES, compute_metrics

RESULTS_DIR = REPO_ROOT / "results"
REPORT_DIR = RESULTS_DIR / "report_rq3"
MASK_CHECK_DIR = RESULTS_DIR / "rq3_mask_check"
SCENARIOS_ORDER = ["orig_to_orig", "lung_to_lung", "out_to_out", "orig_to_lung", "orig_to_out"]
FROZEN_SEED = 42
B0_MACRO_F1 = 0.1627
A0_BEST_MACRO_F1 = 0.2397

print(f"Thư mục gốc dự án: {REPO_ROOT}")
print(f"NumPy: {np.__version__} | Pandas: {pd.__version__}")


## 1. Kiểm tra trực quan & Căn chỉnh hình học Mặt nạ Phổi (S4-07)

- Mặt nạ phổi gốc trong bộ dữ liệu có kích thước `256×256` RGB với hai mức giá trị `{0, 255}` (`results/mask_size_audit.json`, `results/mask_value_audit.json`).
- Quá trình tiền xử lý co giãn mặt nạ phổi lên kích thước ảnh `299×299` bằng nội suy `cv2.INTER_NEAREST` và lấy ngưỡng nhị phân `> 127` để không làm nhòe biên giải phẫu.
- Đã lấy mẫu ngẫu nhiên phân tầng đúng **20 ảnh/lớp** (tổng cộng **80 ảnh**) từ tập `val` (`allow_test=False`), tính toán độ phủ diện tích phổi (`lung_fraction`), toạ độ trọng tâm chuẩn hoá và kiểm tra độ lệch biên.


In [ ]:
mask_summary_path = MASK_CHECK_DIR / "mask_alignment_summary.json"
mask_samples_path = MASK_CHECK_DIR / "mask_visual_check_samples.csv"

assert mask_summary_path.is_file(), f"Thiếu tệp {mask_summary_path}"
assert mask_samples_path.is_file(), f"Thiếu tệp {mask_samples_path}"

mask_summary = json.loads(mask_summary_path.read_text(encoding="utf-8"))
mask_samples_df = pd.read_csv(mask_samples_path)

print("=== KẾT QUẢ KIỂM TOÁN CĂN CHỈNH MẶT NẠ PHỔI S4-07 ===")
print(f"- Số ảnh kiểm tra trực quan : {mask_summary['total_visual_samples']} (20 ảnh/lớp x 4 lớp tập val)")
print(f"- Số mặt nạ rỗng            : {mask_summary['empty_masks_count']}")
print(f"- Số mặt nạ căn giữa hợp lệ  : {mask_summary['centered_masks_count']} ({mask_summary['all_centered']})")
print(f"- Tỷ lệ diện tích phổi TB    : {mask_summary['lung_fraction_mean']:.4f} (Min: {mask_summary['lung_fraction_min']:.4f}, Max: {mask_summary['lung_fraction_max']:.4f})")

print("\n=== THỐNG KÊ HÌNH HỌC MẶT NẠ THEO TỪNG LỚP ===")
cls_stats = mask_samples_df.groupby("label").agg({
    "lung_fraction": ["count", "mean", "std", "min", "max"],
    "centroid_y_norm": "mean",
    "centroid_x_norm": "mean",
    "left_lung_fraction": "mean",
})
print(cls_stats.to_string(float_format=lambda x: f"{x:.4f}"))

print("\n=== DANH SÁCH BẢNG ẢNH TRỰC QUAN CONTACT SHEETS (GỐC + VIỀN | LUNG-ONLY | OUTSIDE-ONLY) ===")
for c_name, fname in mask_summary["contact_sheets"].items():
    p_img = MASK_CHECK_DIR / fname
    assert p_img.is_file(), f"Thiếu ảnh contact sheet {p_img}"
    print(f"  • Lớp {c_name:16s}: {p_img.name} (Tồn tại: {p_img.is_file()})")


## 2. Kiểm toán 5 Run thực nghiệm RQ3 và Cổng hợp lệ H1–H6

Mỗi run trong `results/rq3_ghep_svm_<scenario>_s42` được kiểm tra:
- Tồn tại đủ 4 tệp `config.yaml`, `metrics.json`, `log`, `preds_val.csv` (Cổng H1).
- `allow_test = False`, `n_train = 14.815`, `n_val = 3.174`, mã băm `split_v1.csv` khớp chuẩn (Cổng H2, H6).
- Chuẩn hoá `StandardScaler` chỉ fit trên tập train tương ứng của từng chế độ che (Cổng H6).
- Điểm `macro_f1` tính lại trực tiếp từ `preds_val.csv` khớp với `metrics.json` và kiểm tra xác suất dự đoán (tổng mỗi hàng xấp xỉ 1.0).


In [ ]:
EXPECTED_COMMIT = "9ce35960cac5ea48967ce2ef640c56515ed8933b"
EXPECTED_SPLIT_SHA256 = "f99e03d3d5cfc32f2bbc9befbdef45a07a256458a837a6d420a6658ed7583d98"

audit_rows = []
for sc in SCENARIOS_ORDER:
    run_id = f"rq3_ghep_svm_{sc}_s{FROZEN_SEED}"
    run_dir = RESULTS_DIR / run_id
    for req_file in ("config.yaml", "metrics.json", "log", "preds_val.csv"):
        assert (run_dir / req_file).is_file(), f"Thiếu {req_file} trong {run_id}"

    cfg = yaml.safe_load((run_dir / "config.yaml").read_text(encoding="utf-8"))
    met = json.loads((run_dir / "metrics.json").read_text(encoding="utf-8"))
    preds_df = pd.read_csv(run_dir / "preds_val.csv")

    recomp = compute_metrics(preds_df["y_true"].to_numpy(), preds_df["y_pred"].to_numpy())
    exact_f1 = macro_f1_from_labels(preds_df["y_true"].to_numpy(), preds_df["y_pred"].to_numpy())

    assert cfg["commit"] == EXPECTED_COMMIT
    assert cfg["features"] == "ghep"
    assert cfg["classifier"] == "SVC"
    assert cfg["split"]["sha256"] == EXPECTED_SPLIT_SHA256
    assert cfg["evaluation"]["allow_test"] is False
    assert cfg["preprocessing"]["scaler"] == "StandardScaler"
    assert abs(recomp["macro_f1"] - met["macro_f1"]) < 1e-12
    assert abs(recomp["accuracy"] - met["accuracy"]) < 1e-12
    assert recomp["confusion_matrix"] == met["confusion_matrix"]
    proba_cols = [c for c in preds_df.columns if c.startswith("proba_")]
    assert len(proba_cols) == 4
    row_sums = preds_df[proba_cols].sum(axis=1)
    assert row_sums.min() > 0.9999 and row_sums.max() < 1.0001

    sec = met["seconds"]
    audit_rows.append({
        "run_id": run_id,
        "scenario": sc,
        "train_mode": cfg["extraction"]["train_mode"],
        "eval_mode": cfg["extraction"]["eval_mode"],
        "macro_f1_val": met["macro_f1"],
        "accuracy_val": met["accuracy"],
        "fit_cpu_s": sec["fit_cpu"],
        "predict_wall_s": sec["predict_wall"],
    })

audit_df = pd.DataFrame(audit_rows)
print("Kiểm toán 5/5 run RQ3: ĐẠT TUYỆT ĐỐI (H1-H6).")
print(audit_df.to_string(index=False, float_format=lambda x: f"{x:.4f}"))


## 3. Báo cáo Bắt buộc R1, R2, R4, R5 và Phép so sánh chính RQ3 (`src/bootstrap.py`)

- Gọi `build_rq3_report` tạo/cập nhật toàn bộ các tệp báo cáo chuẩn tại `results/report_rq3/`.
- Trình bày **Bảng R1 Summary** kèm khoảng tin cậy 95% bootstrap 2.000 lần.
- Trình bày **Bảng Phép so sánh chính RQ3** (`lung_to_lung` vs `orig_to_orig`) theo Mục 5 của `threshold_v3.md`.
- Trình bày **Bảng Kiểm tra lối tắt R3** đối chứng đặc trưng Trong phổi vs Ngoài phổi (`lung_to_lung` vs `out_to_out`) và Toàn ảnh vs Ngoài phổi (`orig_to_orig` vs `out_to_out`).


In [ ]:
report_paths = build_rq3_report(
    results_dir=RESULTS_DIR,
    out_dir=REPORT_DIR,
    scenarios=SCENARIOS_ORDER,
    seed=FROZEN_SEED,
    reference="orig_to_orig",
    n_boot=2000,
    rng_seed=42,
    expected_n=3174,
    threshold_path=RESULTS_DIR / "threshold_v3.md",
)

r1_summary_df = pd.read_csv(report_paths["r1_summary.csv"])
r1_boot_df = pd.read_csv(report_paths["r1_bootstrap_vs_reference.csv"])
r3_shortcut_df = pd.read_csv(report_paths["r3_shortcut_bootstrap.csv"])
r2_df = pd.read_csv(report_paths["r2_normal_lung_opacity.csv"])
r4_df = pd.read_csv(report_paths["r4_sampling_error.csv"])
r5_text_content = report_paths["r5_gioi_han.md"].read_text(encoding="utf-8").strip()

print("=== BẢNG R1 SUMMARY: MACRO-F1 VÀ RECALL TỪNG LỚP TRÊN 5 KỊCH BẢN (95% CI BOOTSTRAP 2.000 LẦN) ===")
cols_r1 = [
    "scenario",
    "macro_f1",
    "ci95_low",
    "ci95_high",
    "recall_COVID",
    "recall_Lung_Opacity",
    "recall_Normal",
    "recall_Viral Pneumonia",
]
print(r1_summary_df[cols_r1].to_string(index=False, float_format=lambda x: f"{x:.4f}"))

print("\n=== PHÉP SO SÁNH CHÍNH RQ3 (MỤC 5 THRESHOLD_V3): LUNG_TO_LUNG vs ORIG_TO_ORIG ===")
primary_row = r3_shortcut_df[
    (r3_shortcut_df["scenario_a"] == "lung_to_lung") & (r3_shortcut_df["scenario_b"] == "orig_to_orig")
].iloc[0]
passes_primary = (float(primary_row["point_diff"]) >= 0.005) and (not bool(primary_row["contains_zero"]))
print(f"  Hiệu số Macro-F1 (point_diff) = {primary_row['point_diff']:+.6f} (Ngưỡng yêu cầu >= +0.005)")
print(f"  95% CI Bootstrap ghép cặp     = [{primary_row['ci_low']:+.6f}, {primary_row['ci_high']:+.6f}] (contains_zero = {primary_row['contains_zero']})")
print(f"  => KẾT LUẬN QUY TẮC: Đạt điều kiện thay thế = {passes_primary}")
print(f"  => QUYẾT ĐỊNH CẤU HÌNH ĐÓNG BĂNG ĐI TIẾP VÀO TẬP TEST (S5-04): '{'lung_to_lung' if passes_primary else 'orig_to_orig'}'")

print("\n=== BẢNG R3: KIỂM ĐỊNH LỐI TẮT SHORTCUT LEARNING GIỮA CÁC CẶP TƯƠNG PHẢN (BOOTSTRAP 2.000 LẦN) ===")
print(r3_shortcut_df[["scenario_a", "scenario_b", "point_diff", "ci_low", "ci_high", "contains_zero"]].to_string(
    index=False, float_format=lambda x: f"{x:+.6f}"
))

print("\n=== BẢNG R2: MA TRẬN CON 2x2 NORMAL - LUNG OPACITY TRÊN 5 KỊCH BẢN ===")
print(r2_df[["scenario", "LO_to_LO", "LO_to_Normal", "Normal_to_LO", "Normal_to_Normal"]].to_string(index=False))

print("\n=== BẢNG R4: SAI SỐ CHUẨN LẤY MẪU XẤP XỈ ===")
print(r4_df[["scenario", "p_accuracy", "se_all", "p_recall_viral", "se_viral"]].to_string(index=False, float_format=lambda x: f"{x:.4f}"))

print(f"\n=== R5: GIỚI HẠN PHƯƠNG PHÁP LUẬN ===\n{r5_text_content}")


## 4. Phân tích Độ quan trọng Hoán vị Khối (Block Permutation Importance — S5-02)

Đánh giá mức độ sụt giảm Macro-F1 ($\Delta \text{Macro-F1}$) khi hoán vị giá trị của từng khối đặc trưng trên 3 mô hình đã huấn luyện:
1. `original`: Huấn luyện và đánh giá trên ảnh gốc.
2. `lung_only`: Huấn luyện và đánh giá trên ảnh chỉ giữ vùng phổi.
3. `outside_only`: Huấn luyện và đánh giá trên ảnh chỉ giữ vùng ngoài phổi.


In [ ]:
perm_csv_path = RESULTS_DIR / "rq3_permutation_importance.csv"
assert perm_csv_path.is_file(), f"Thiếu tệp {perm_csv_path}"
perm_df = pd.read_csv(perm_csv_path)

for m_name in ["original", "lung_only", "outside_only"]:
    sub_perm = perm_df[perm_df["mask_mode"] == m_name].sort_values("importance_mean", ascending=False)
    print(f"=== ĐỘ QUAN TRỌNG HOÁN VỊ KHỐI ĐẶC TRƯNG: {m_name.upper()} (S5-02) ===")
    print(sub_perm[["block_name", "n_dims", "baseline_score", "importance_mean", "importance_std"]].to_string(
        index=False, float_format=lambda x: f"{x:.4f}"
    ))
    print()


## 5. Kết luận Khoa học về Shortcut Learning và Quyết định Đóng băng (S5-03)

### 5.1. Bằng chứng thực nghiệm xác nhận Shortcut Learning mạnh mẽ:
1. **Mô hình ngoài phổi (`out_to_out`) đạt Macro-F1 = 0,8921:**
   - Khi hoàn toàn xóa đen 100% nhu mô phổi, mô hình vẫn đạt độ chính xác 87,81% và Macro-F1 = 0,8921.
   - So sánh `orig_to_orig` vs `out_to_out`: $\Delta = +0,0103$, khoảng tin cậy 95% `[-0,0017, +0,0220]` **chứa 0**! Về mặt thống kê, mô hình phân loại trên toàn ảnh không vượt trội hơn so với một mô hình chỉ nhìn phông nền và khung xương ngoài phổi.
2. **Căn nguyên y sinh học:**
   - Lớp `Viral Pneumonia` đạt **Recall = 96,55%** trên mô hình `out_to_out` (cao nhất trong cả 5 kịch bản). Lý do: 100% ảnh lớp này là ảnh bệnh nhi 1–5 tuổi (nguồn Kaggle Paul Mooney), có lồng ngực nhỏ, xương sườn mảnh và tư thế chụp trẻ em.
   - Lớp `COVID` đạt **Recall = 92,99%** trên `out_to_out` nhờ nắm bắt các ký hiệu thiết bị chụp kỹ thuật số (`DX`, `CX`), ký hiệu chữ in `L/R` ngoài thành ngực từ các nguồn BIMCV, SIRM, Hannover.
3. **Tín hiệu bệnh lý thực sự trong phổi (`lung_to_lung`):**
   - Đạt Macro-F1 = 0,8490 (thấp hơn `orig_to_orig` -0,0535 do mất các lối tắt nền).
   - Permutation importance chỉ ra khối **GLCM (kết cấu mờ đục và thâm nhiễm)** đóng vai trò quyết định sống còn ($\Delta = 0,4960$), đặc biệt là `glcm_dissimilarity_12` ($\Delta = 0,4655$) và `glcm_contrast_12` ($\Delta = 0,3163$).

### 5.2. Quyết định đóng băng cấu hình cho Tuần 10 (S5-04):
- Theo Mục 5 `threshold_v3.md`, vì `lung_to_lung` không cải thiện $\ge +0,005$ so với `orig_to_orig`, nhóm quyết định **giữ nguyên cấu hình ảnh gốc `orig_to_orig`** (`SVC(RBF)` trên `ghep` 152 chiều, `seed=42`) cho bài kiểm tra một lần duy nhất trên tập `test` ở Tuần 10 (`S5-04`).
- Toàn bộ các phát hiện về Shortcut Learning được lưu trữ chính thức tại `results/rq3_shortcut_learning_conclusion.md` và sẽ là nội dung cảnh báo y khoa trọng tâm trong Báo cáo tổng kết (`S6-01`).
